In [1]:
import numpy as np
import math

batch_size = 2 #Two sentences at once
seq_len = 3
d_model = 8
num_heads = 2
d_k = d_model // num_heads # 4

# Our input X (batch_size, seq_len, d_model) -> (2, 3, 8)
X = np.random.randn(batch_size, seq_len, d_model)

# The massive weight matrices (d_model, d_model) -> (8, 8)
# Notice they project d_model to d_model, not d_model to d_k!
W_Q = np.random.randn(d_model, d_model)
W_K = np.random.randn(d_model, d_model)
W_V = np.random.randn(d_model, d_model)
W_O = np.random.randn(d_model, d_model) # The final output projection (Section 3.2)

In [10]:
# Project X to get Q, K, V

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

# Reshape to include num_heads
Q_reshaped = Q.reshape(batch_size, seq_len, num_heads, d_k)
K_reshaped = K.reshape(batch_size, seq_len, num_heads, d_k)
V_reshaped = V.reshape(batch_size, seq_len, num_heads, d_k)

# Transpose to get the num_heads to a before axis
Q_reshaped_t = Q_reshaped.transpose(0,2,1,3)
K_reshaped_t = K_reshaped.transpose(0,2,1,3)
V_reshaped_t = V_reshaped.transpose(0,2,1,3)

# Calculating score
S = Q_reshaped_t @ K_reshaped_t.transpose(0,1,3,2)
S = S / (math.sqrt(d_k))

# Applying Softmax

S_max_sub = S - np.max(S, axis=-1, keepdims=True)
S_max_sub_epow = np.exp(S_max_sub)
S_sum = np.sum(S_max_sub_epow, axis=-1, keepdims=True)
S_normalised = S_max_sub_epow/S_sum

# Calculate output

output = S_normalised @ V_reshaped_t

# Reshape output

output_t = output.transpose(0,2,1,3)
output_reshaped = output_t.reshape(batch_size, seq_len, d_model)

# Final Y

Y = output_reshaped @ W_O

In [21]:
# Applying residual connections

X_residual = X + Y
print(f"Shape of X_residual is {X_residual.shape}")

# Layer Norm
epsilon = 1e-5
gamma = np.ones(d_model)
beta = np.zeros(d_model)

X_residual_mean = np.mean(X_residual, axis=-1, keepdims=True)
X_residual_var = np.var(X_residual, axis=-1, keepdims=True)
X_centered = (X_residual - X_residual_mean) / np.sqrt(X_residual_var + epsilon)
X_norm = gamma * X_centered + beta

# Verify norm
print("---Verify Norm---")
print(np.mean(X_norm[0, 0, :]))
print(np.var(X_norm[0, 0, :]))
print("---Verify Norm End---")

Shape of X_residual is (2, 3, 8)
---Verify Norm---
0.0
0.9999998848083074
---Verify Norm End---
